# Molekylvisualisering

```{admonition} Læringsutbytte
Etter å ha arbeidet med dette temaet, skal du kunne:

1. hente molekyl- og proteinstrukturer fra PubChem og Protein Data Bank
2. tegne molekyler i tre dimensjoner med `py3Dmol` og velge mellom ulike representasjoner
3. legge på molekylflater og forklare hva overflaten og fargene representerer
4. tegne proteiner med `nglview` og velge ut bestemte deler av strukturen
5. visualisere en molekyldynamikksimulering som en animasjon
6. bygge en 3D-struktur fra en SMILES-kode med RDKit og vise den
7. velge et egnet visualiseringsbibliotek for en bestemt oppgave
```

Så langt har vi i stor grad representert molekyler med tekst og tall, for eksempel SMILES-koder, molekylformler og tabeller med deskriptorer. Tredimensjonal visualisering gir oss en annen type informasjon: Vi kan undersøke molekylform, plasseringen av funksjonelle grupper, aktive seter i enzymer og hvordan ulike deler av et biomolekyl er organisert.

En visualisering er en modell som fremhever noen egenskaper og skjuler andre. En pinnemodell viser bindingene tydelig, mens en kalottmodell gir et bedre inntrykk av hvor stor plass atomene tar. Valg av representasjon er derfor en del av den faglige tolkningen.

I dette kapitlet bruker vi bibliotekene `py3Dmol` og `nglview`. Begge kan vise tredimensjonale strukturer i en notebook, men de har ulike styrker og bruksområder.


## Installasjon

Kjør kodecellen nedenfor én gang for å installere bibliotekene.

`nglview` er en **widget**. Det betyr at den interaktive figuren er koblet til en kjørende Python-kjerne, i stedet for å være et vanlig statisk bilde. Dette får betydning når notebooken skal deles eller bygges som nettside.

Eldre veiledninger kan inneholde ekstra kommandoer for å aktivere `nglview` som en notebook-utvidelse. I nyere Jupyter-miljøer er dette vanligvis ikke nødvendig.


In [ ]:
!pip install py3Dmol nglview rdkit mdtraj

## Hvor kommer strukturene fra?

Bibliotekene kan hente strukturer fra to store, åpne databaser:

- **[PubChem](https://pubchem.ncbi.nlm.nih.gov/)** inneholder strukturer og egenskaper for små molekyler, for eksempel legemidler, naturstoffer og industrikjemikalier. Hver forbindelse har en **CID** (Compound ID).
- **[Protein Data Bank](https://www.rcsb.org/)** inneholder strukturer av proteiner, nukleinsyrer og molekylkomplekser. Hver struktur har en firetegns **PDB-ID**, for eksempel `1PSN` eller `4HHB`.

Det er viktig å skille mellom en eksperimentelt bestemt struktur og en struktur som er generert fra en SMILES-kode. En PDB-struktur bygger vanligvis på røntgendiffraksjon, kryoelektronmikroskopi eller NMR og har en tilhørende usikkerhet og oppløsning. En 3D-struktur som genereres gjennom SMILES, er derimot en beregnet modell. Begge kan være nyttige, men de representerer ulike typer kunnskap.

## Visualisering med py3Dmol

`py3Dmol` er enkelt å komme i gang med. Vi henter en struktur, velger en representasjon og viser figuren.


In [15]:
import py3Dmol

paracetamol = py3Dmol.view(query="cid:1983")     # PubChem CID
paracetamol.setStyle({"stick": {"colorscheme": "cyanCarbon"}})
#paracetamol.zoomTo()
paracetamol.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Figuren kan roteres med musa, zoomes med rullehjulet og flyttes med høyre musetast.

Representasjonen angis med `setStyle`, som tar imot en dictionary. Nøkkelen angir representasjonstypen, mens verdien inneholder innstillingene. Noen vanlige typer er:

| Type | Viser | Egner seg til |
|---|---|---|
| `line` | tynne streker | store systemer og rask oversikt |
| `stick` | pinnemodell | små og mellomstore molekyler |
| `sphere` | kalottmodell | å vise hvor mye plass atomene tar |
| `cartoon` | bånd og piler | proteiner og nukleinsyrer |
| `cross` | kryss ved hvert atom | atomposisjoner uten tydelige bindinger |

Vi viser den samme forbindelsen med tre representasjoner for å sammenlikne hvilken informasjon de fremhever.


In [8]:
visning = py3Dmol.view(query="cid:2519")

visning.setStyle({"line": {}},                              viewer=(0, 0))
visning.setStyle({"stick": {"colorscheme": "cyanCarbon"}},  viewer=(0, 1))
visning.setStyle({"sphere": {"scale": 0.9}},                viewer=(0, 2))

visning.zoomTo()
visning.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
import py3Dmol
py3Dmol.view.__module__  # sjekk at pakka faktisk er importert

'py3Dmol'

```{admonition} Underveisoppgave: Samme molekyl, ulike representasjoner
:class: tip

Studer de tre fremstillingene av koffein.

1. Hvilken gir best oversikt over hvilke atomer som er bundet sammen?
2. Hvilken viser tydeligst hvor stor plass molekylet tar?
3. Hvilken ville du brukt for å forklare hvorfor koffein er tilnærmet plant?
4. Bytt CID-en med et molekyl du har arbeidet med tidligere i emnet, og gjenta vurderingen.

En representasjon fremhever alltid noen egenskaper og skjuler andre. Valget bør derfor begrunnes ut fra hva figuren skal brukes til.
```


### Proteiner

For store biomolekyler er `cartoon` en nyttig representasjon. Atomene skjules, og ryggraden vises som bånd. Da blir sekundærstrukturen tydelig: alfahelikser vises som spiraler og betaplater som brede piler.


In [ ]:
rna_polymerase = py3Dmol.view(query="pdb:5IYC", width=600, height=450)
rna_polymerase.setStyle({"cartoon": {"color": "spectrum"}})
rna_polymerase.zoomTo()
rna_polymerase.show()


Fargevalget `spectrum` gir en gradvis fargeendring langs kjeden og gjør det lettere å følge polypeptidryggraden.

Vi kan også kombinere flere representasjoner. I eksemplet nedenfor vises hemoglobin som bånd, hemgruppene som pinnemodeller og jernionene som kuler. Da blir plasseringen av bindingsstedene for oksygen tydelig.


In [ ]:
hemoglobin = py3Dmol.view(query="pdb:4HHB", width=600, height=450)

hemoglobin.setStyle({"cartoon": {"color": "spectrum"}})
hemoglobin.addStyle({"resn": "HEM"}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.2}})
hemoglobin.addStyle({"resn": "HEM", "elem": "Fe"}, {"sphere": {"radius": 0.8, "color": "orange"}})

hemoglobin.zoomTo()
hemoglobin.show()


Legg merke til forskjellen mellom `setStyle` og `addStyle`. `setStyle` setter hovedrepresentasjonen, mens `addStyle` legger en ny representasjon oppå den eksisterende for et valgt sett atomer. Utvalget angis med en dictionary: `{"resn": "HEM"}` betyr alle atomene som tilhører en rest med navnet HEM.

```{admonition} Underveisoppgave: Finn hemgruppen
:class: tip

1. Hvor mange hemgrupper ser du i hemoglobin? Stemmer dette med det du vet om proteinets struktur?
2. Zoom inn på én hemgruppe. Hvilket grunnstoff ligger i sentrum, og hva er koordinasjonstallet?
3. Bytt ut `4HHB` med `1MBN` (myoglobin). Hva er den viktigste strukturelle forskjellen mellom hemoglobin og myoglobin, og hvordan henger forskjellen sammen med funksjonen?
```


### Molekylflater og fargelegging

En molekylflate gir et inntrykk av formen og størrelsen til molekylet. En van der Waals-flate bygges opp fra van der Waals-radiene til atomene og viser omtrent hvilket volum molekylet opptar.

Flaten kan fargelegges på ulike måter, for eksempel etter hvilket atom som ligger nærmest overflaten. Fargene må alltid tolkes ut fra fargeregelen som er valgt; de representerer ikke automatisk elektrisk potensial eller ladningsfordeling.


In [ ]:
elektronkart = py3Dmol.view(query="cid:12180", width=500, height=400)  # metylbutanoat

elektronkart.setStyle({"stick": {"colorscheme": "Jmol"}})
elektronkart.addSurface("VDW", {
    "opacity": 0.65,
    "colorscheme": "Jmol",
})
elektronkart.zoomTo()
elektronkart.show()


```{admonition} Ikke forveksle farge med elektrisk potensial
:class: warning

I figuren ovenfor er overflaten fargelagt etter atomtype. Rødt viser områder nær oksygenatomer, mens grått viser områder nær karbonatomer. Fargene er altså en visualiseringsregel, ikke en beregning av elektronfordelingen.

Et virkelig elektrostatisk potensialkart krever egne potensialdata, for eksempel fra en kvantekjemisk beregning eller en volumetrisk datafil. `py3Dmol` kan vise slike data, men beregner dem ikke selv.
```

```{admonition} Underveisoppgave: Hva viser overflaten?
:class: tip

Lag en pinnemodell med en halvgjennomsiktig van der Waals-flate for:

1. vann (CID 962)
2. metan (CID 297)
3. etanol (CID 702)
4. eddiksyre (CID 176)

Sammenlikn figurene. Hvilke atomer er tilgjengelige på overflaten? Hvordan endres formen når molekylet blir større? Diskuter deretter hvordan struktur og tilgjengelige funksjonelle grupper kan påvirke intermolekylære krefter.
```


### Fra SMILES til 3D

I forrige kapittel genererte vi en tredimensjonal konformasjon med RDKit og lagret den som en molblokk. Den samme strukturen kan vises med `py3Dmol`.

RDKit lager strukturen, mens `py3Dmol` viser den. Bibliotekene kan utveksle informasjon fordi begge støtter det samme standardiserte formatet.


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem


def lag_3d(smiles, seed=42):
    molekyl = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(molekyl, randomSeed=seed)
    AllChem.MMFFOptimizeMolecule(molekyl)
    return Chem.MolToMolBlock(molekyl)


blokk = lag_3d("CC(C)Cc1ccc(cc1)C(C)C(=O)O")   # ibuprofen

visning = py3Dmol.view(width=500, height=400)
visning.addModel(blokk, "mol")
visning.setStyle({"stick": {"colorscheme": "cyanCarbon"}})
visning.zoomTo()
visning.show()


```{admonition} Underveisoppgave: Hva slags modell får vi?
:class: tip

Strukturen ovenfor er beregnet med et kraftfelt, ikke bestemt eksperimentelt.

1. Kjør `lag_3d` med tre ulike verdier for `seed` og vis alle tre. Blir de like? Hva forteller det deg?
2. Hent den samme forbindelsen fra PubChem med `query="cid:3672"` og sammenlikn med strukturen du genererte. Hvor stemmer de, og hvor gjør de det ikke?
3. Prøv med et molekyl med en lang, fleksibel kjede, for eksempel en fettsyre. Hvordan endrer svaret på spørsmål 1 seg, og hvorfor?
4. Formuler i én setning hva `EmbedMolecule` faktisk gir deg, og hva den *ikke* gir deg.
```

````{admonition} Løsningsforslag
:class: tip, dropdown

1. For et lite, ganske stivt molekyl blir konformasjonene nokså like, men ikke identiske. `EmbedMolecule` bruker en tilfeldig startgjetning, og `seed` styrer den.

2. PubChem-strukturen er også en beberegnet konformasjon, ikke en måling. De to strukturene vil ha svært like bindingslengder og vinkler, men kan ha ulike torsjonsvinkler rundt enkeltbindinger.

3. For en fleksibel kjede spriker resultatene mye mer. Molekylet har mange lavtliggende konformasjoner med omtrent samme energi, og kraftfeltet finner bare *en* av dem.

4. `EmbedMolecule` gir deg **én rimelig konformasjon** med fornuftige bindingslengder og vinkler. Den gir deg *ikke* den mest stabile konformasjonen, og heller ikke fordelingen av konformasjoner som molekylet faktisk har i løsning. Til det trenger du en konformasjonssøking eller en molekyldynamikksimulering.
````


## Visualisering med nglview

`nglview` gir omfattende kontroll over hvilke deler av en struktur som skal vises. En viktig funksjon er **seleksjonsspråket**, som brukes til å velge bestemte kjeder, aminosyrerester, ligander eller atomgrupper. Dette er særlig nyttig for proteiner.

Biblioteket er også utviklet for **trajektorier**, altså serier av strukturer som viser hvordan et system endrer seg over tid. Det egner seg derfor godt til molekyldynamikksimuleringer.

Vi begynner med pepsin, et enzym i magesekken som bryter proteiner ned til kortere polypeptider.


In [ ]:
import nglview as nv

enzym = nv.show_pdbid("1PSN")
enzym.layout.width = "600px"
enzym.layout.height = "450px"
enzym


Legg merke til at den siste linjen bare inneholder variabelnavnet. Widgeten vises fordi objektet er den siste verdien i cellen, på samme måte som en dataframe kan vises uten `print`.

### Seleksjonsspråket

En seleksjon er en tekststreng som beskriver hvilke atomer eller rester som skal velges:

| Seleksjon | Betyr |
|---|---|
| `protein` | alle aminosyrerester |
| `water` | vannmolekyler |
| `hetero` | komponenter som ikke er protein eller nukleinsyre, for eksempel ligander, ioner og kofaktorer |
| `GLY` | alle glysinrester |
| `TRP or TYR or PHE` | alle aromatiske aminosyrerester |
| `1-50` | rest nummer 1 til 50 |
| `:A` | kjede A |
| `backbone` | ryggraden |
| `sidechainAttached` | sidekjedene |

Vi kan legge en halvgjennomsiktig overflate på hele proteinet eller bare på utvalgte rester.


In [ ]:
enzym_overflate = nv.show_pdbid("1PSN")
enzym_overflate.add_surface(selection="protein", opacity=0.3)
enzym_overflate.layout.width = "600px"
enzym_overflate.layout.height = "450px"
enzym_overflate


In [ ]:
# Bare glysinrestene får overflate. Da ser du hvor de ligger i strukturen.
enzym_glysin = nv.show_pdbid("1PSN")
enzym_glysin.add_surface(selection="GLY", opacity=0.4, color="tomato")
enzym_glysin.layout.width = "600px"
enzym_glysin.layout.height = "450px"
enzym_glysin


Vi kan også vise bestemte sidekjeder som pinnemodeller oppå båndrepresentasjonen. Her fremhever vi tryptofanrestene, som har store aromatiske sidekjeder.


In [ ]:
enzym_trp = nv.show_pdbid("1PSN")
enzym_trp.add_licorice("TRP")          # TRP er tryptofan
enzym_trp.center(selection="TRP")
enzym_trp.layout.width = "600px"
enzym_trp.layout.height = "450px"
enzym_trp


Hold musepekeren over en rest for å se navnet og nummeret. Dette er nyttig når du utforsker en struktur du ikke kjenner fra før.

### Bytte representasjon

`clear_representations` fjerner de eksisterende representasjonene. Deretter kan figuren bygges opp på nytt med ønskede utvalg og stiler.


In [ ]:
enzym_kule_pinne = nv.show_pdbid("1PSN")
enzym_kule_pinne.clear_representations()
enzym_kule_pinne.add_representation("ball+stick", selection="protein")
enzym_kule_pinne.layout.width = "600px"
enzym_kule_pinne.layout.height = "450px"
enzym_kule_pinne


```{admonition} Underveisoppgave: Det aktive setet
:class: tip

1. Vis pepsin (`1PSN`) og legg på en halvgjennomsiktig overflate over hele proteinet. Roter strukturen til du finner den dype kløften i overflaten. Det er det aktive setet.
2. Pepsin er en aspartatprotease, og har to katalytisk aktive asparaginsyrerester i bunnen av kløften. Vis dem med `add_licorice("ASP")` og se om du finner dem. Hvorfor tror du de ligger akkurat der?
3. Pepsin skilles ut som proenzymet **pepsinogen**, og endrer struktur ved den lave pH-en i magesekken. Vis pepsinogen (`3PSG`) med samme oppsett. Hva er den viktigste forskjellen du ser?
4. Ut fra strukturen: tror du pepsinogen kan ha samme virkning som pepsin? Begrunn med det du ser.
```

````{admonition} Løsningsforslag
:class: tip, dropdown

2. De to asparaginsyrerestene ligger i bunnen av kløften fordi det er der substratet, altså polypeptidkjeden som skal kuttes, må passere. Katalyse krever at de reaktive gruppene er i kontakt med akkurat den bindingen som skal brytes.

3. Pepsinogen har et ekstra segment på omtrent 44 aminosyrer i den ene enden. Det ligger som en propp inne i kløften og dekker det aktive setet.

4. Nei. Så lenge proppen sitter der, kommer ikke substratet frem til de katalytiske restene. Ved lav pH endrer ladningsfordelingen seg, propp-segmentet løsner og spaltes av, og enzymet blir aktivt. Dette er en generell mekanisme: mange fordøyelsesenzymer og enzymer i blodkoagulasjon skilles ut som inaktive forstadier, slik at de ikke bryter ned vevet der de dannes.
````


### Molekyler fra RDKit

`nglview` kan også vise et RDKit-molekyl direkte, uten at strukturen først må konverteres til en molblokk.


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

koffein = Chem.AddHs(Chem.MolFromSmiles("CN1C=NC2=C1C(=O)N(C)C(=O)N2C"))
AllChem.EmbedMolecule(koffein, randomSeed=42)
AllChem.MMFFOptimizeMolecule(koffein)

visning = nv.show_rdkit(koffein)
visning


### Statiske bilder

Den interaktive widgeten er nyttig til utforsking, men i en rapport eller presentasjon trenger vi ofte et statisk bilde. `nglview` kan rendre den aktuelle visningen som et bilde.

Kommandoene må kjøres i **to separate celler**. Renderingen skjer i nettleseren, og bildet er først tilgjengelig når den første cellen er ferdig.


In [ ]:
enzym_trp.render_image()


In [ ]:
enzym_trp._display_image()


## Visualisering av simuleringer

Så langt har vi arbeidet med enkeltstrukturer. I en **molekyldynamikksimulering (MD)** beregnes bevegelsen til atomene over mange små tidssteg.

Resultatet lagres som en **trajektorie**: en serie strukturer som representerer systemet ved ulike tidspunkter. Trajektorier kan blant annet brukes til å studere fleksibilitet, konformasjonsendringer og hvordan molekyler beveger seg i forhold til hverandre.

Her skal vi ikke kjøre selve simuleringen, men lese inn og visualisere en eksisterende trajektorie. Biblioteket `nglview` inneholder demofiler som kan brukes til dette.

For å lese trajektoriefiler trenger vi et bibliotek som støtter filformatet. To vanlige alternativer er `mdtraj` og `MDAnalysis`.


In [ ]:
import nglview as nv
import mdtraj as md

trajektorie = md.load(nv.datafiles.TRR, top=nv.datafiles.PDB)
print(trajektorie)

animasjon = nv.show_mdtraj(trajektorie)
animasjon.layout.width = "600px"
animasjon.layout.height = "450px"
animasjon


Trykk på avspillingsknappen under figuren. Skyvefeltet lar deg gå til et bestemt tidssteg.

En trajektorie kan også leses med `MDAnalysis`:

```{code-block} python
import MDAnalysis as mda
from MDAnalysis.tests.datafiles import PSF, DCD

univers = mda.Universe(PSF, DCD)
protein = univers.select_atoms("protein")

animasjon = nv.show_mdanalysis(protein)
animasjon
```

```{admonition} Demodataene ligger i en egen pakke
:class: warning

`MDAnalysis.tests.datafiles` er *ikke* en del av `MDAnalysis` selv. Den ligger i pakken `MDAnalysisTests`, som du må installere separat med `pip install MDAnalysisTests`.

Dette er en klassisk snublestein. Feilmeldinga sier at modulen ikke finnes, og det er lett å tro at man har installert feil bibliotek.
```

```{admonition} Underveisoppgave: Se på bevegelsen
:class: tip

1. Kjør animasjonen ovenfor. Hvilke deler av strukturen beveger seg mest, og hvilke ligger nesten stille? Hva tror du forklarer forskjellen?
2. Legg på en overflate, og se animasjonen på nytt. Blir det lettere eller vanskeligere å se hva som skjer? Hva sier det om valg av representasjon?
3. Hvor mange rammer inneholder trajektorien? Bruk `len(trajektorie)`. Hvis hver ramme er lagret hvert 10. pikosekund, hvor lang tid dekker simuleringen?
4. Sammenlikn den tiden med tiden det tar for et enzym å utføre en katalytisk syklus, som kan være fra mikrosekunder til millisekunder eller lengre. Hva er den praktiske konsekvensen av det forholdet for hva MD-simuleringer kan og ikke kan si noe om?
```


## Hvilket bibliotek bør du velge?

Bibliotekene overlapper, men har ulike styrker.

| | py3Dmol | nglview |
|---|---|---|
| Små molekyler | svært godt egnet | godt egnet |
| Proteiner | godt egnet | svært godt egnet |
| Velge deler av en struktur | enkle utvalg | omfattende seleksjonsspråk |
| Trajektorier og animasjon | begrenset | utviklet for dette |
| Flere figurer side om side | `viewergrid` | vanligvis ett vindu om gangen |
| Molekylflater | innebygget | innebygget med flere valgmuligheter |
| Avhengigheter | få | flere, inkludert widget-systemet |
| Visning på en statisk nettside | fungerer ofte direkte | krever ofte ekstra oppsett eller ny kjøring |

`py3Dmol` skriver HTML og JavaScript til output-cellen. Når notebooken lagres, kan den interaktive figuren ofte følge med og vises uten en aktiv Python-kjerne.

`nglview` er en Jupyter-widget. Visningen er tettere koblet til notebookmiljøet, og på en statisk nettside kan den kreve innebygget widgettilstand eller at brukeren kjører cellen selv.

```{admonition} Praktisk tommelfingerregel
:class: tip

Skal figuren kunne leses av andre uten at de kjører koden, er `py3Dmol` ofte det enkleste valget.

Skal du utforske en struktur eller en trajektorie interaktivt, gir `nglview` vanligvis flere muligheter.
```


## Sluttoppgaver

```{admonition} Oppgave 1: Bindingsforhold du kan se
:class: tip

Velg tre molekyler som illustrerer ulike bindingsforhold: ett med bare enkeltbindinger, ett med en dobbeltbinding, og ett aromatisk.

1. Vis alle tre som pinnemodeller side om side med `viewergrid`.
2. Legg på en halvgjennomsiktig van der Waals-flate for hvert molekyl.
3. Skriv en kort tekst der du forklarer forskjellene du ser, knyttet til molekylform, atomtyper og hvilke funksjonelle grupper som er tilgjengelige på overflaten.
4. Generer de samme tre molekylene fra SMILES med RDKit og sammenlikn geometrien med strukturene fra PubChem. Er noen av dem systematisk forskjellige? Hvorfor?
```

```{admonition} Oppgave 2: Et enzym og substratet
:class: tip

Velg et enzym fra PDB som er løst sammen med et substrat, en substratanalog eller en hemmer. Lysozym med en sukkerkjede (`1HEW`) er et godt utgangspunkt, men velg gjerne noe fra ditt eget fagfelt.

1. Vis proteinet som bånd og liganden som pinnemodell.
2. Bruk seleksjonsspråket i `nglview` til å vise sidekjedene som ligger nær liganden.
3. Legg på en halvgjennomsiktig overflate og vis at liganden ligger i en lomme.
4. Lag et statisk bilde og skriv en figurtekst på tre til fire setninger som forklarer hva figuren viser.
5. Slå opp den katalytiske mekanismen til enzymet og forklar hvilke av restene du ser som er direkte involvert.
```

```{admonition} Oppgave 3: Konformasjoner og energi
:class: tip

Butan har en velkjent energiprofil for rotasjon rundt den midterste karbon-karbon-bindingen.

1. Bygg butan i RDKit og generer 20 ulike konformasjoner med `AllChem.EmbedMultipleConfs`.
2. Beregn energien til hver konformasjon med `AllChem.MMFFOptimizeMoleculeConfs`.
3. Lag et histogram over energiene. Hvor mange energiminima ser du?
4. Vis den laveste og den høyeste konformasjonen i `py3Dmol` og sammenlikn dem.
5. Sammenlikn med den teoretiske energiprofilen for butan fra læreboka. Stemmer forholdet mellom anti og gauche?
```

```{admonition} Oppgave 4: Din egen figur til en rapport
:class: tip

Lag én figur som du kunne brukt i en labrapport eller en presentasjon. Den skal:

1. Vise et molekyl eller protein som er relevant for noe du faktisk har gjort på laben.
2. Bruke minst to representasjoner i samme figur.
3. Ha en bevisst begrunnelse for hvert valg av representasjon og farge.
4. Lagres som et statisk bilde.

Lever figuren sammen med en kort tekst på fem til ti setninger der du begrunner valgene dine, og der du sier hva figuren *ikke* viser. Det siste er like viktig som det første.
```
